# Lab: preparing the Titanic for a model

Lab of the lesson [Data Preprocessing](https://insper.github.io/ML/2026.2/classes/preprocessing/): from the raw Titanic CSV to a numeric matrix with no holes, no text and comparable scales, every statistic learned on the training rows only. The notebook follows the two labs of the page: the **fundamental** lab (steps 1 to 6) first, then the **advanced** one (steps 7 to 9), in the same session. Each step opens with a **Predict**: answer it before running the cell. The solutions of the simulations are on the page.

## Fundamental lab — preparing the Titanic, column by column

### Step 1 — load and take inventory

**Predict.** Before looking: which columns of a passenger list do you expect to have holes? Which ones are text, and which numbers are not really quantities?

**Resposta (antes de rodar):** Pela experiência com listas de passageiros de navios antigos, espero
buracos em `Age` (idade nem sempre era registrada) e em `Cabin` (só quem tinha cabine reservada aparece) e
talvez algum em `Embarked` (porto de embarque, erro de preenchimento). Colunas de texto puro: `Name`, `Sex`,
`Ticket`, `Cabin`, `Embarked`. Números que não são realmente quantidades: `PassengerId` (é só um rótulo),
`Survived` (0/1 é categoria, não magnitude) e `Pclass` (1/2/3 é uma categoria ordinal, não uma medida em que
3 é "três vezes" 1).

In [1]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/hsandmann/biblio/refs/heads/main/ml/aula04/titanic/train.csv"
df = pd.read_csv(url)
print(df.shape)
inventory = pd.DataFrame({'numeric': [pd.api.types.is_numeric_dtype(df[c]) for c in df.columns],
                          'missing': df.isna().sum(),
                          'distinct': df.nunique()})
print(inventory)

(891, 12)
             numeric  missing  distinct
PassengerId     True        0       891
Survived        True        0         2
Pclass          True        0         3
Name           False        0       891
Sex            False        0         2
Age             True      177        88
SibSp           True        0         7
Parch           True        0         7
Ticket         False        0       681
Fare            True        0       248
Cabin          False      687       147
Embarked       False        2         3


**Resposta (depois de rodar):** Confirmado quase integralmente. O inventário mostra 891 linhas × 12
colunas, com buracos em `Age` (177 faltantes), `Cabin` (687 faltantes) e `Embarked` (2 faltantes) — exatamente
as três colunas que eu esperava. `Name` tem 891 valores distintos (um por passageiro), `Ticket` 681 e `Cabin`
147, confirmando que são texto/identificadores e não quantidades. `PassengerId`, `Survived` e `Pclass` aparecem
como `numeric=True` no inventário (são `int64`), mas continuam sendo rótulos/categorias disfarçados de número,
como eu previ.

### Step 2 — split before anything else

**Predict.** The split comes before any imputation or scaling. What would go wrong if you computed the median age on all 891 passengers and only then split?

**Resposta (antes de rodar):** Se a mediana da idade fosse calculada nas 891 linhas e só depois os dados
fossem divididos, as estatísticas de preenchimento "veriam" os próprios passageiros do conjunto de teste antes do
treino acontecer — isso é *data leakage*: a mediana usada para preencher o treino conteria informação do teste,
e a avaliação final ficaria otimista demais (o modelo pareceria generalizar melhor do que de fato generaliza).

In [2]:
from sklearn.model_selection import train_test_split

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X, y = df[features], df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print("missing in train:", X_train.isna().sum()[lambda s: s > 0].to_dict())
print("missing in test: ", X_test.isna().sum()[lambda s: s > 0].to_dict())

(712, 7) (179, 7)
missing in train: {'Age': 137, 'Embarked': 2}
missing in test:  {'Age': 40}


**Resposta (depois de rodar):** Confirmado. O treino tem 712 linhas e o teste 179 (proporção 80/20
mantendo a taxa de sobrevivência, por `stratify=y`). Os buracos aparecem separadamente em cada conjunto:
`Age` falta em 137 linhas de treino e 40 de teste; `Embarked` falta em 2 linhas, só no treino. Como o split vem
antes de qualquer imputação, qualquer estatística (mediana, média, moda) que formos calcular a partir daqui usará
só as 712 linhas de treino — exatamente o que evita o vazamento que descrevi na resposta anterior.

### Step 3 — why is Age missing?

**Predict.** Do you expect age to be missing equally often in the three classes? And would deleting the passengers without an age change what the training set looks like?

**Resposta (antes de rodar):** Não espero que a idade falte igualmente nas três classes — passageiros
de 3ª classe provavelmente tinham registros mais incompletos (bilhetes mais baratos, menos burocracia) do que os
de 1ª classe. Um `dropna` nessas linhas tende a remover desproporcionalmente passageiros de 3ª classe, então o
treino que sobra ficaria com uma composição de classes diferente da original (mais inclinado para 1ª/2ª classe).

In [3]:
missing = X_train['Age'].isna()
print("share of missing Age by class:", missing.groupby(X_train['Pclass']).mean().round(3).to_dict())
print("survival rate, Age present / missing:", y_train.groupby(missing).mean().round(3).to_dict())
kept = X_train.dropna(subset=['Age'])
print(f"dropna keeps {len(kept)} of {len(X_train)} rows; "
      f"3rd-class share {(X_train['Pclass'] == 3).mean():.3f} -> {(kept['Pclass'] == 3).mean():.3f}")

share of missing Age by class: {1: 0.152, 2: 0.06, 3: 0.261}
survival rate, Age present / missing: {False: 0.405, True: 0.292}
dropna keeps 575 of 712 rows; 3rd-class share 0.549 -> 0.503


**Resposta (depois de rodar):** Confirmado. A fração de `Age` ausente por classe é {1: 0.152, 2: 0.06,
3: 0.261} — 3ª classe tem quase o dobro da taxa de buracos da 1ª e mais de quatro vezes a da 2ª, como eu esperava.
A taxa de sobrevivência também difere entre quem tem idade registrada (0.405) e quem não tem (0.292), sinal de que
a ausência não é aleatória (MCAR), está ligada a sobrevivência e classe. E o `dropna` confirma o viés: ele mantém
575 das 712 linhas de treino, e a participação da 3ª classe cai de 0.549 para 0.503 — exatamente a distorção que
eu previ, perdendo desproporcionalmente passageiros mais pobres.

### Step 4 — compare imputation strategies for Age

**Predict.** After filling 137 holes with one value, will the standard deviation of the ages go up, stay the same or go down? Which age will become the most frequent one?

**Resposta (antes de rodar):** Preencher 137 buracos com um único valor constante reduz a variabilidade
da coluna — o desvio-padrão deve cair (os valores observados variam, o valor imputado não varia nada). A idade
mais frequente depois da imputação passa a ser o próprio valor usado para preencher (média ou mediana), porque
ele se repete 137 vezes de uma só vez, batendo qualquer pico natural que já existisse nos dados observados.

In [4]:
from sklearn.impute import SimpleImputer

cols = ['Age', 'Pclass', 'SibSp', 'Parch']
imputers = {'mean': SimpleImputer(strategy='mean'),
            'median': SimpleImputer(strategy='median')}
observed = X_train['Age'].dropna()
rows = {'observed only': [len(observed), observed.mean(), observed.std(), observed.min(),
                          observed.value_counts().iloc[0]]}
for name, imp in imputers.items():
    ages = pd.Series(imp.fit(X_train[cols]).transform(X_train[cols])[:, 0])
    rows[name] = [len(ages), ages.mean(), ages.std(), ages.min(), ages.value_counts().iloc[0]]
print(pd.DataFrame(rows, index=['n', 'mean', 'std', 'min', 'largest pile']).T.round(2))

                   n   mean    std   min  largest pile
observed only  575.0  29.81  14.49  0.42          26.0
mean           712.0  29.81  13.02  0.42         137.0
median         712.0  29.56  13.03  0.42         139.0


**Resposta (depois de rodar):** Confirmado nos dois pontos. Nas 575 idades observadas o desvio-padrão é
14.49; depois de imputar as 137 faltantes com a média (29.81) ele cai para 13.02, e com a mediana (29.56) cai para
13.03 — queda de quase 1.5 ponto em ambos os casos, como eu previ. E o "maior pico" (`largest pile`) salta de 26
(contagem natural de alguma idade comum nos dados observados) para 137 com a média e 139 com a mediana — ou seja,
o próprio valor imputado se torna disparadamente o mais frequente, confirmando a segunda parte da hipótese.

**Simulation: the shrinkage law, on holes you make.** Take the 575 real ages of the training rows and erase 20%, 50% and 80% of them at random (MCAR). Fill the holes with the mean and compare the standard deviation with the law of section 6, the standard deviation of the kept ages times \(\sqrt{(n_{\text{kept}} - 1)/(n - 1)}\). Does the law hold every time? How far does the filled column drift from the 14.49 of the real ages?

In [5]:
rng = np.random.default_rng(0)
real = X_train['Age'].dropna()                                  # the 575 ages we do have
print(f"all 575 real ages: std {real.std():.2f}")
for erase in [0.2, 0.5, 0.8]:
    holed = real.copy()
    holed[rng.random(len(real)) < erase] = np.nan               # erase ages at random (MCAR)
    filled = SimpleImputer(strategy='mean').fit_transform(holed.to_frame()).ravel()
    n_kept = holed.notna().sum()
    law = holed.std() * np.sqrt((n_kept - 1) / (len(holed) - 1))
    print(f"erase {erase:.0%}: {n_kept} ages kept, std {holed.std():.2f} · after mean imputation "
          f"{filled.std(ddof=1):.2f} · law {law:.2f}")

all 575 real ages: std 14.49
erase 20%: 463 ages kept, std 14.65 · after mean imputation 13.15 · law 13.15
erase 50%: 285 ages kept, std 13.98 · after mean imputation 9.83 · law 9.83
erase 80%: 120 ages kept, std 13.62 · after mean imputation 6.20 · law 6.20


**Resposta:** A lei se confirma nas três rodadas: o desvio-padrão depois da imputação pela média bate,
com duas casas decimais, com a previsão `std(mantidas) * sqrt((n_mantidas-1)/(n-1))` em todos os níveis de
apagamento (20%, 50% e 80%). O afastamento em relação ao desvio-padrão real das 575 idades completas (14.49) cresce
com a fração apagada: a 20% o desvio cai para perto de 13 (ainda moderado), a 50% cai para a faixa de 9-10 (perda
clara de informação) e a 80% despenca para a faixa de 6 (menos da metade do valor real) — quanto menos dados
observados sobram, mais a imputação por um valor único "achata" a variabilidade real da coluna.

### Step 5 — encode Sex, Embarked and Pclass

**Predict.** Three categorical columns with 2, 3 and 3 levels: how many one-hot columns will they become, and how many 1s will each row have?

**Resposta (antes de rodar):** `Sex` tem 2 níveis, `Embarked` 3 e `Pclass` 3 (tratado aqui como
categórico): 2 + 3 + 3 = 8 colunas one-hot no total. Cada linha deve ter exatamente três 1s — um 1 por coluna
original codificada (uma marcação para o sexo, uma para o porto, uma para a classe).

In [6]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = ['Sex', 'Embarked', 'Pclass']
cat_imp = SimpleImputer(strategy='most_frequent').set_output(transform='pandas')
cats_train = cat_imp.fit_transform(X_train[cat_cols])            # learns the modes on train
cats_test = cat_imp.transform(X_test[cat_cols])
print("modes learned on train:", cat_imp.statistics_)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(cats_train)
print(ohe.get_feature_names_out())
print(X_test[cat_cols].head(3).to_numpy().tolist())
print(ohe.transform(cats_test.head(3)))

modes learned on train: ['male' 'S' 3]
['Sex_female' 'Sex_male' 'Embarked_C' 'Embarked_Q' 'Embarked_S' 'Pclass_1'
 'Pclass_2' 'Pclass_3']
[['male', 'S', 3], ['male', 'S', 3], ['male', 'C', 3]]
[[0. 1. 0. 0. 1. 0. 0. 1.]
 [0. 1. 0. 0. 1. 0. 0. 1.]
 [0. 1. 1. 0. 0. 0. 0. 1.]]


**Resposta (depois de rodar):** Confirmado. As modas aprendidas no treino (usadas para preencher os 2
buracos de `Embarked`) são `['male', 'S', 3]`. `ohe.get_feature_names_out()` devolve 8 colunas (2 de `Sex` + 3 de
`Embarked` + 3 de `Pclass`), exatamente como eu previ. E cada linha da matriz transformada (`ohe.transform(...)`)
soma 3, um 1 em cada um dos três blocos — confirmando a segunda parte da hipótese.

### Step 6 — scale the numeric columns

**Predict.** In raw units, which numeric column will own the distance between two passengers? And after scaling with the training statistics, will the test columns have mean exactly 0?

**Resposta (antes de rodar):** Em unidades brutas, `Fare` (que varia de 0 a mais de 500) deve dominar a
distância ao quadrado entre dois passageiros, muito mais do que `Age`, `SibSp` ou `Parch` (todas em escalas de
dezenas ou unidades). Depois de escalar com as estatísticas do treino, a média do teste **não** deve ser
exatamente 0 — o `StandardScaler` aprendeu média e desvio do treino, e o teste é uma amostra diferente, então por
acaso sua própria média/desvio serão parecidos mas não idênticos a 0/1.

In [7]:
from sklearn.preprocessing import StandardScaler

num_cols = ['Age', 'SibSp', 'Parch', 'Fare']
num_imp = SimpleImputer(strategy='median').set_output(transform='pandas').fit(X_train[num_cols])
num_train, num_test = num_imp.transform(X_train[num_cols]), num_imp.transform(X_test[num_cols])
scaler = StandardScaler().set_output(transform='pandas').fit(num_train)
z_train, z_test = scaler.transform(num_train), scaler.transform(num_test)

share = lambda M: (100 * M.var() / M.var().sum()).round(1)     # share of the squared distance
print(pd.DataFrame({'raw %': share(num_train), 'scaled %': share(z_train)}))
print(pd.DataFrame({'train mean': z_train.mean(), 'train std': z_train.std(ddof=0),
                    'test mean': z_test.mean(), 'test std': z_test.std(ddof=0)}).round(3))

       raw %  scaled %
Age      6.8      25.0
SibSp    0.0      25.0
Parch    0.0      25.0
Fare    93.1      25.0
       train mean  train std  test mean  test std
Age           0.0        1.0     -0.036     0.995
SibSp        -0.0        1.0      0.141     1.178
Parch        -0.0        1.0     -0.053     0.791
Fare         -0.0        1.0      0.040     1.160


**Resposta (depois de rodar):** Confirmado nos dois pontos. Em unidades brutas `Fare` responde por 93.1%
da distância ao quadrado (`Age` 6.8%, `SibSp` e `Parch` ~0.0%) — domina quase completamente, como eu previ. Depois
de escalar, as quatro colunas passam a contribuir igualmente, 25.0% cada (por definição do `StandardScaler`, que
normaliza a variância de cada coluna de treino para 1). E, como também previ, a média/desvio do teste não ficam
exatamente em 0/1: médias de -0.036 (Age), 0.141 (SibSp), -0.053 (Parch) e 0.040 (Fare), e desvios de 0.995, 1.178,
0.791 e 1.160 — próximos de 0/1, mas não exatos, porque as estatísticas vieram do treino, e isso é o comportamento
correto (é exatamente o que evita o vazamento).

**Simulation: whose neighbors?** Take the first test passenger and find the 5 training passengers closest to them on the four numeric columns, once in raw units (`num_train`, `num_test`) and once scaled (`z_train`, `z_test`). How many passengers appear in both lists? What do the neighbors of each list have in common?

In [8]:
from sklearn.metrics import pairwise_distances

first = X_test.index[0]                                         # the first test passenger
print("test passenger", first, num_test.loc[first].to_dict())
nearest = {}
for name, A, q in [('raw', num_train, num_test.loc[[first]]), ('scaled', z_train, z_test.loc[[first]])]:
    d = pairwise_distances(q, A).ravel()
    nearest[name] = A.index[np.argsort(d)[:5]]                  # the 5 nearest training passengers
    print(f"{name} units, 5 nearest:")
    print(num_train.loc[nearest[name]].to_string())
print("in both lists:", len(set(nearest['raw']) & set(nearest['scaled'])))

test passenger 565 {'Age': 24.0, 'SibSp': 2.0, 'Parch': 0.0, 'Fare': 24.15}
raw units, 5 nearest:
      Age  SibSp  Parch  Fare
316  24.0    1.0    0.0  26.0
600  24.0    2.0    1.0  27.0
312  26.0    1.0    1.0  26.0
551  27.0    0.0    0.0  26.0
874  28.0    1.0    0.0  24.0
scaled units, 5 nearest:
      Age  SibSp  Parch     Fare
301  28.5    2.0    0.0  23.2500
330  28.5    2.0    0.0  23.2500
48   28.5    2.0    0.0  21.6792
69   26.0    2.0    0.0   8.6625
392  28.0    2.0    0.0   7.9250
in both lists: 0


**Resposta:** As duas listas de 5 vizinhos não têm nenhum passageiro em comum. Em unidades brutas, o
vizinho mais próximo é decidido quase inteiramente pela tarifa (`Fare`) parecida — a lista "raw" reúne passageiros
com `Fare` próximo do passageiro de teste, mesmo que a idade ou o tamanho da família sejam bem diferentes. Já a
lista "scaled" reúne passageiros parecidos de forma equilibrada nas quatro colunas (idade, irmãos/cônjuges, pais/
filhos e tarifa, todas com o mesmo peso depois do `StandardScaler`). Como as duas noções de "parecido" são
diferentes, não é surpresa que não compartilhem nenhum vizinho — é a mesma lição do passo 6, agora vista caso a
caso em vez de em médias.

**To close the fundamental part.** In one sentence each: (1) why is a k-NN on raw Titanic columns, in practice, a k-NN on fare? (2) Where does the median of step 4 come from, and what do the test passengers' missing ages receive? (3) Why are the scaled test columns of step 6 not exactly mean 0 and standard deviation 1, and why is that correct?

**Resposta (fundamental):**

1. Porque, em unidades brutas, `Fare` concentra 93.1% da distância ao quadrado entre dois passageiros (passo 6) —
as outras três colunas numéricas (`Age`, `SibSp`, `Parch`) contribuem quase nada. Um k-NN que mede distância
euclidiana direta nessas colunas está, na prática, só comparando tarifas.
2. A mediana do passo 4 vem só das 575 idades observadas nas 712 linhas de **treino** (o `SimpleImputer` é
ajustado em `X_train`); os passageiros de teste com idade faltante recebem exatamente esse mesmo valor (29.56),
nunca uma estatística calculada a partir do próprio teste.
3. Porque o `StandardScaler` aprendeu média e desvio-padrão só do treino; o teste é uma amostra diferente da
população, então sua média/desvio empíricos vão ser *parecidos* com 0/1 mas não idênticos (médias entre -0.053 e
0.141, desvios entre 0.791 e 1.178). Isso é correto porque é exatamente a garantia contra vazamento: se
escalássemos o teste com as próprias estatísticas, ele ficaria artificialmente centrado, e deixaríamos de medir se
o modelo generaliza para dados realmente novos.

### Your turn (fundamental)

At home:

1. **Another scaler.** Redo step 6 with `RobustScaler` instead of `StandardScaler`. Print its `scale_`: which columns get a spread of exactly 1, and why, for each one? How do the shares of the squared distance change?
2. **Fare's shape.** Compute the skewness of `X_train['Fare']`, of `np.log1p(X_train['Fare'])` and of the logged column after a `StandardScaler`. Which step changes the shape, and which does not?
3. **Pclass as ordinal.** In step 5, leave `Pclass` out of the one-hot and keep it as a single column (1, 2, 3). How many columns does the encoding produce now, and what does this choice assume about the three classes?

In [9]:
from sklearn.preprocessing import RobustScaler

robust = RobustScaler().set_output(transform='pandas').fit(num_train)
rz_train, rz_test = robust.transform(num_train), robust.transform(num_test)
print("RobustScaler.scale_:", dict(zip(num_cols, robust.scale_.round(3))))
print(pd.DataFrame({'raw %': share(num_train), 'standard %': share(z_train), 'robust %': share(rz_train)}))

RobustScaler.scale_:

 {'Age': np.float64(14.0), 'SibSp': np.float64(1.0), 'Parch': np.float64(1.0), 'Fare': np.float64(23.104)}
       raw %  standard %  robust %
Age      6.8        25.0      12.3
SibSp    0.0        25.0      16.0
Parch    0.0        25.0      10.0
Fare    93.1        25.0      61.6


**Resposta:** Duas colunas recebem `scale_` exatamente igual a 1 com o `RobustScaler`: `SibSp` e `Parch`.
O `RobustScaler` usa o IQR (percentil 75 − percentil 25) como escala, e para essas duas colunas discretas, onde a
maioria dos passageiros tem 0 ou 1 irmão/cônjuge e 0 ou 1 pai/filho a bordo, o percentil 25 e o percentil 75 caem
exatamente 1 unidade um do outro (ex.: Q1=0, Q3=1) — por isso o IQR, e portanto `scale_`, é exatamente 1.0, e a
coluna sai do `RobustScaler` praticamente sem ser reescalada. `Age` (IQR de 14 anos) e `Fare` (IQR de 23.1) são
contínuas e têm espalhamento muito maior, então ficam com `scale_` de 14.0 e 23.104, respectivamente. Como a
escala de cada coluna muda de forma desigual, as participações na distância ao quadrado (`robust %`) deixam de
ficar divididas em 25% cada como no `StandardScaler`: `Fare` volta a dominar (61.6%, mesmo depois de escalada),
`SibSp` (16.0%) e `Age` (12.3%) ficam no meio, e `Parch` cai para 10.0% — porque dividir por um IQR igual a 1 não
encolhe `SibSp`/`Parch` tanto quanto o `StandardScaler` (que usa o desvio-padrão, maior que 1) encolhia.

In [10]:
from scipy.stats import skew

fare_raw = X_train['Fare']
fare_log = np.log1p(fare_raw)
fare_log_scaled = StandardScaler().fit_transform(fare_log.to_frame()).ravel()
print(f"skewness: raw {skew(fare_raw):.2f} · log1p {skew(fare_log):.2f} · log1p+StandardScaler {skew(fare_log_scaled):.2f}")

skewness: raw 4.64 · log1p 0.31 · log1p+StandardScaler 0.31


**Resposta:** A assimetria (*skewness*) da tarifa bruta é 4.64 — bem alta. Aplicar `log1p` derruba a
assimetria para 0.31 (a transformação é a que muda a *forma* da distribuição). Em seguida, aplicar
`StandardScaler` sobre a coluna já logada mantém a assimetria em 0.31, idêntica à de `log1p` sem escalar — escalar
é só uma transformação linear (subtrai a média, divide pelo desvio), e transformações lineares não alteram o
formato (assimetria/curtose) de uma distribuição, só seu centro e sua escala. Ou seja: quem resolve a forma é o
`log1p` (4.64 → 0.31); o `StandardScaler` resolve a escala, não a forma (0.31 → 0.31, sem mudança).

In [11]:
from sklearn.preprocessing import OneHotEncoder as _OHE

cat_cols_ordinal = ['Sex', 'Embarked']                            # Pclass fica de fora do one-hot
cat_imp_ordinal = SimpleImputer(strategy='most_frequent').set_output(transform='pandas')
cats_train_ordinal = cat_imp_ordinal.fit_transform(X_train[cat_cols_ordinal])
ohe_ord = _OHE(handle_unknown='ignore', sparse_output=False).fit(cats_train_ordinal)
n_cols_ordinal = len(ohe_ord.get_feature_names_out()) + 1          # + a própria coluna Pclass, numérica
print("colunas com Pclass ordinal:", n_cols_ordinal, "vs. colunas com Pclass one-hot:", len(ohe.get_feature_names_out()))

colunas com Pclass ordinal: 6 vs. colunas com Pclass one-hot: 8


**Resposta:** Mantendo `Pclass` como uma única coluna numérica (1, 2 ou 3) em vez de 3 colunas one-hot,
a codificação das três categóricas passa a ter 2 (`Sex`) + 3 (`Embarked`) + 1 (`Pclass`) = 6 colunas, duas a menos
que as 8 do passo 5. Tratar `Pclass` como ordinal/numérico assume que a distância entre classe 1 e 2 é igual à
distância entre classe 2 e 3 (ambas "custam" 1 unidade no espaço de distância), e que a ordem importa de forma
linear — uma suposição razoável aqui, já que 1ª > 2ª > 3ª é mesmo uma hierarquia, mas que impõe um espaçamento
igual que o one-hot não impõe (o one-hot trata as três classes como categorias sem ordem nem distância relativa
definida).

## Advanced lab — one object, a black-box model and smarter imputers

### Step 7 — everything in one object: a ColumnTransformer preview

**Predict.** Steps 5 and 6 used four fitted objects. Bundled into one, how many columns will the output have, and will any `NaN` be left in the test rows?

**Resposta (antes de rodar):** O número de colunas de saída deve continuar sendo o mesmo de antes: 4
numéricas (depois de imputadas e escaladas, uma coluna cada) + 8 do one-hot de `Sex`/`Embarked`/`Pclass` = 12
colunas — o `ColumnTransformer` só empacota os mesmos passos em um único objeto, não cria nem remove colunas. E
não deve sobrar nenhum `NaN` no teste, porque os imputadores de cada ramo (numérico e categórico) são ajustados no
treino e aplicados (transform) no teste, preenchendo qualquer buraco que exista lá.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

preprocess = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                          OneHotEncoder(handle_unknown='ignore')), cat_cols)])
M_train = preprocess.fit_transform(X_train)       # every statistic is learned here, on train only
M_test = preprocess.transform(X_test)             # ... and only applied here
print(M_train.shape, M_test.shape, "NaN left in test:", int(np.isnan(M_test).sum()))
print(preprocess.get_feature_names_out())
print(M_test[:2].round(2))

(712, 12) (179, 12) NaN left in test: 0
['num__Age' 'num__SibSp' 'num__Parch' 'num__Fare' 'cat__Sex_female'
 'cat__Sex_male' 'cat__Embarked_C' 'cat__Embarked_Q' 'cat__Embarked_S'
 'cat__Pclass_1' 'cat__Pclass_2' 'cat__Pclass_3']
[[-0.43  1.42 -0.47 -0.16  0.    1.    0.    0.    1.    0.    0.    1.  ]
 [ 1.11 -0.47  0.73 -0.33  0.    1.    0.    0.    1.    0.    0.    1.  ]]


**Resposta (depois de rodar):** Confirmado. `M_train.shape` é (712, 12) e `M_test.shape` é (179, 12) —
exatamente as 12 colunas que eu previ (4 numéricas + 8 one-hot). E `NaN left in test` é 0: nenhum buraco restante,
porque o `SimpleImputer` de cada ramo foi ajustado em `X_train` dentro do `fit_transform` e só aplicado
(`transform`) no teste.

### Step 8 — does it matter? A black-box k-NN with and without scaling

**Predict.** The baseline that always answers "did not survive" gets 61.5%. Will a k-NN on unscaled columns be far above it? How much will scaling add?

**Resposta (antes de rodar):** Sem escalar, eu esperaria uma melhora pequena sobre os 61.5% do *dummy*,
porque (como vimos no passo 6) a distância bruta é dominada por `Fare`, que por si só já carrega algum sinal de
classe social e sobrevivência, mas ignora idade e família. Escalar deve adicionar uma melhora bem mais substancial,
porque devolve peso a `Age`, `SibSp` e `Parch`, que também se relacionam com sobrevivência (mulheres/crianças
primeiro, famílias pequenas).

In [13]:
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, RobustScaler

baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print(f"baseline (always 'did not survive')   test accuracy {baseline.score(X_test, y_test):.3f}")
for name, scaler in [('no scaling', 'passthrough'), ('StandardScaler', StandardScaler()),
                     ('MinMaxScaler', MinMaxScaler()), ('RobustScaler', RobustScaler())]:
    prep = ColumnTransformer([
        ('num', make_pipeline(SimpleImputer(strategy='median'), scaler), num_cols),
        ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                              OneHotEncoder(handle_unknown='ignore')), cat_cols)])
    knn = make_pipeline(prep, KNeighborsClassifier(n_neighbors=5)).fit(X_train, y_train)
    print(f"k-NN (black box), {name:15s}     test accuracy {knn.score(X_test, y_test):.3f}")

baseline (always 'did not survive')   test accuracy 0.615


k-NN (black box), no scaling          test accuracy 0.676


k-NN (black box), StandardScaler      test accuracy 0.821
k-NN (black box), MinMaxScaler        test accuracy 0.804


k-NN (black box), RobustScaler        test accuracy 0.771


**Resposta (depois de rodar):** Confirmado. O k-NN sem escalar chega a 0.676 — só 6.1 pontos percentuais
acima do *baseline* de 0.615, uma melhora pequena como eu previ. Escalar adiciona uma melhora bem maior: com
`StandardScaler` a acurácia sobe para 0.821 (+14.5 p.p. sobre "sem escalar" e +20.6 p.p. sobre o *baseline*), com
`MinMaxScaler` para 0.804 e com `RobustScaler` para 0.771. O `StandardScaler` é o melhor dos três, e a diferença
entre escalar ou não (mais de 14 pontos) é bem maior do que a diferença entre os escaladores entre si (até 5
pontos) — a decisão que mais importa é escalar, não qual escalador escolher.

**Simulation: shape before scale.** Fare is the most skewed column. Give it a branch of its own in the column transformer: median, then `FunctionTransformer(np.log1p)`, then `StandardScaler`, with the other numeric columns in the usual branch. Does the skewness of the training fares drop? Does the k-NN of step 8 improve?

In [14]:
from sklearn.preprocessing import FunctionTransformer

rest = ['Age', 'SibSp', 'Parch']
logfare = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), rest),
    ('fare', make_pipeline(SimpleImputer(strategy='median'), FunctionTransformer(np.log1p),
                           StandardScaler()), ['Fare']),                 # shape, then scale
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                          OneHotEncoder(handle_unknown='ignore')), cat_cols)])

print(f"skewness of the training fares: raw {skew(X_train['Fare']):.2f} · log1p {skew(np.log1p(X_train['Fare'])):.2f}")

num_train_log = num_train.copy()
num_train_log['Fare'] = np.log1p(num_train_log['Fare'])
print("share of the squared distance, log1p(Fare) unscaled:", share(num_train_log).round(1).to_dict())

knn_log = make_pipeline(logfare, KNeighborsClassifier(n_neighbors=5)).fit(X_train, y_train)
print(f"k-NN, log1p(Fare) then StandardScaler   test accuracy {knn_log.score(X_test, y_test):.3f}")

skewness of the training fares: raw 4.64 · log1p 0.31
share of the squared distance, log1p(Fare) unscaled: {'Age': 98.4, 'SibSp': 0.7, 'Parch': 0.4, 'Fare': 0.6}


k-NN, log1p(Fare) then StandardScaler   test accuracy 0.782


**Resposta:** O `log1p` resolve o problema de forma que o passo 4 apontava: a assimetria da tarifa de
treino cai de um valor alto (perto de 4.6-4.8) para perto de 0.3-0.4, quase simétrica. Sem reescalar, essa coluna
logada some do radar da distância — depois do `log1p` (sem `StandardScaler`), `Age` passa a responder por cerca de
98% da distância ao quadrado e `Fare` cai para menos de 1%, o oposto do passo 6 (onde `Fare` bruta dominava com
93%). Mas isso não se traduz em um k-NN melhor: a acurácia de teste cai para 0.782, sete passageiros piores em 179
do que os 0.821 do `StandardScaler` sem log. Corrigir a forma (assimetria) não é o mesmo que melhorar o modelo —
achatar os valores extremos de tarifa also apaga parte do sinal (tarifas muito altas = passageiros muito ricos,
muito mais prováveis de sobreviver) que o k-NN original estava explorando.

### Step 9 — imputers that use the other columns

**Predict.** An imputer that looks at class and family size: will its imputed ages spread more or less than the median's? Can it produce an impossible age?

**Resposta (antes de rodar):** Um imputador que usa `Pclass`, `SibSp` e `Parch` para prever a idade
(KNN ou iterativo) deveria produzir idades com mais variação entre si do que a mediana (que sempre repete o mesmo
número), porque ele adapta o valor a cada combinação de classe/família — então o desvio-padrão deveria subir,
ficando mais perto dos 14.49 observados do que os ~13.0 da mediana. Sim, pode produzir uma idade impossível: um
regressor linear como o `IterativeImputer`, sem nenhum limite, pode extrapolar para valores negativos se a
combinação de colunas estiver fora do que foi visto no treino.

In [15]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer

others = {'knn': KNNImputer(n_neighbors=5),
          'iterative': IterativeImputer(random_state=0, min_value=0)}
for name, imp in others.items():
    ages = pd.Series(imp.fit(X_train[cols]).transform(X_train[cols])[:, 0])
    rows[name] = [len(ages), ages.mean(), ages.std(), ages.min(), ages.value_counts().iloc[0]]
print(pd.DataFrame(rows, index=['n', 'mean', 'std', 'min', 'largest pile']).T.round(2))

                   n   mean    std   min  largest pile
observed only  575.0  29.81  14.49  0.42          26.0
mean           712.0  29.81  13.02  0.42         137.0
median         712.0  29.56  13.03  0.42         139.0
knn            712.0  29.22  14.05  0.42          75.0
iterative      712.0  29.54  13.46  0.00          75.0


**Resposta (depois de rodar):** Parcialmente confirmado. O desvio-padrão sobe com os dois imputadores mais
"inteligentes": mediana 13.03, KNNImputer 14.05, IterativeImputer 13.46 — mais perto dos 14.49 reais, como eu
previ (o KNN chega bem mais perto do valor real do que a mediana). Sobre idades impossíveis: nesta célula o mínimo
fica em 0.42 (KNN) e 0.00 (iterativo) — nenhum valor negativo —, mas só porque o código já usa
`IterativeImputer(random_state=0, min_value=0)`, com o limite inferior travado em 0. Sem esse `min_value`, a
segunda parte da minha hipótese se confirma: a versão sem limite do `IterativeImputer` (vista na página da aula)
chega a um mínimo de -6.26, com 7 idades negativas entre as 712 — exatamente a extrapolação impossível que eu
previ.

> **Nota:** a página da aula publica para o `KNNImputer` mean 29.99 e std 13.36, enquanto esta execução (com o
> mesmo código, mesmo `cols` e mesmo `n_neighbors=5`) produz mean 29.22 e std 14.05 — mesmo `n`, mesmo `min` e
> mesma `largest pile` (75), só a média/desvio mudam. A causa mais provável é a versão do scikit-learn instalada
> neste ambiente (1.9.1): o `KNNImputer` não fixa semente nem tem aleatoriedade própria, mas o critério de
> desempate/ponderação entre vizinhos com a mesma distância já mudou entre versões da biblioteca em outras
> ocasiões, o que bastaria para deslocar a média sem alterar o mínimo nem a contagem de linhas. Não alterei
> parâmetro algum do enunciado para tentar "bater" com a página.

**Simulation: a smarter imputer inside the pipeline.** In the numeric branch of the step-8 pipeline (with `StandardScaler`), replace the median by `KNNImputer(n_neighbors=5)`, once on raw units and once between two scalers (the imputer measures distances, so section 12 says it needs comparable scales). Does the k-NN's test accuracy improve?

In [16]:
from sklearn.impute import KNNImputer as _KNNImp

variants = {
    'median, then scale': make_pipeline(SimpleImputer(strategy='median'), StandardScaler()),
    'KNNImputer on raw units, then scale': make_pipeline(_KNNImp(n_neighbors=5), StandardScaler()),
    'scale, KNNImputer, scale again': make_pipeline(StandardScaler(with_std=False),
                                                     _KNNImp(n_neighbors=5), StandardScaler()),
}
for name, num_branch in variants.items():
    prep = ColumnTransformer([
        ('num', num_branch, num_cols),
        ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                              OneHotEncoder(handle_unknown='ignore')), cat_cols)])
    knn_v = make_pipeline(prep, KNeighborsClassifier(n_neighbors=5)).fit(X_train, y_train)
    print(f"{name:38s} test accuracy {knn_v.score(X_test, y_test):.3f}")

median, then scale                     test accuracy 0.821


KNNImputer on raw units, then scale    test accuracy 0.804


scale, KNNImputer, scale again         test accuracy 0.799


**Resposta:** Nenhuma das duas versões com `KNNImputer` supera a mediana simples. "Mediana, depois
escalar" fica em 0.821; "`KNNImputer` em unidades brutas, depois escalar" cai para 0.804 (cerca de 3 passageiros
piores em 179); "escalar, `KNNImputer`, escalar de novo" cai mais, para 0.799 (cerca de 4 passageiros piores). As
diferenças são de 3 a 4 passageiros num teste de 179 — dentro do que a própria página trata como ruído de amostra,
não uma vantagem real. Isso reforça a resposta do passo 9: um imputador mais sofisticado capta melhor a
variabilidade real da coluna, mas isso não garante um modelo melhor, porque o k-NN final é sensível a detalhes
finos da escala e da ordem das transformações, não só à qualidade estatística da idade imputada.

> **Nota:** a página publica 0.810 para a variante "KNNImputer em unidades brutas, depois escalar"; esta execução
> produz 0.804 (1 passageiro de diferença em 179, isto é, uma troca de classificação). É consistente com a mesma
> divergência de versão do `KNNImputer` já registrada na resposta do passo 9 — o imputador produz idades
> levemente diferentes das da página, o que pode mudar o rótulo de 1 passageiro no k-NN final.

**To close the lesson.** In one sentence each: (1) what does a `ColumnTransformer` guarantee that four separately fitted objects do not? (2) Why can a log that fixes the shape of the fare still make the k-NN worse? (3) Why does an imputer that keeps more of the spread not necessarily give a better model?

**Resposta (avançado):**

1. Um `ColumnTransformer` garante que o mesmo `fit` aprenda, de uma só vez e só sobre o treino, todas as
estatísticas de todos os ramos (medianas, modas, médias/desvios, categorias vistas), e que o mesmo `transform`
aplique essas estatísticas na mesma ordem de colunas sempre — no passo 7 isso veio junto com shape consistente
(712/179 linhas × 12 colunas) e zero `NaN` no teste. Quatro objetos ajustados separadamente dependem de o
programador lembrar de chamar `fit` só no treino e `transform` (nunca `fit_transform`) no teste, toda vez, na
ordem certa — um erro manual nessa disciplina é o tipo de vazamento que o passo 2 descreveu.
2. Porque a assimetria de uma coluna é uma propriedade estatística da *forma* da distribuição, não uma medida de
quanto ela ajuda a prever o alvo. Na simulação "shape before scale", o `log1p` deixou `Fare` quase simétrica
(assimetria de 4.64 para 0.31), mas também comprimiu justamente os valores altos de tarifa que sinalizavam
passageiros ricos com maior sobrevivência — a acurácia do k-NN caiu de 0.821 para 0.782 porque o sinal útil (não a
forma) foi parcialmente apagado.
3. Porque manter mais variabilidade (desvio-padrão mais alto, como os 14.05/13.46 do KNNImputer/IterativeImputer
contra os 13.03 da mediana) só ajuda se essa variabilidade adicional refletir a relação real entre as colunas
usadas para imputar e o alvo. Na simulação do passo 9, nenhuma das versões "mais realistas" do imputador superou a
mediana simples (0.821 contra 0.804 e 0.799) — o k-NN final não se beneficiou da imputação mais sofisticada,
porque a ordem/escala das transformações importa mais para ele do que o quão "real" é cada idade imputada.

### Your turn (advanced)

At home, with more time:

1. **The missing flag.** Use `SimpleImputer(strategy='median', add_indicator=True)` in the numeric branch. How many columns does the output have now, and what is the name of the new one? Does the k-NN of step 8 change?
2. **Titles.** Extract the title from `Name` (`df['Name'].str.extract(r',\s*([^.]+)\.')`), add it to the features and to `cat_cols`, and encode it with `OneHotEncoder(handle_unknown='ignore', min_frequency=10)`. Then impute `Age` with the **median age of each title**, learned on the training rows (`groupby('Title')['Age'].median()`). What does it give to the boys titled *Master* whose age is missing (3 of them in the training rows), compared with the global median of 28.5?
3. **The leak, in numbers.** Fit `preprocess` on all 891 rows instead of `X_train`. Compare the learned median age and the fare mean and standard deviation (`preprocess.named_transformers_['num']`) with the train-only version. Which statistic moves most, and why is that the one that depends on a few extreme rows?

In [17]:
prep_flag = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), num_cols),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                          OneHotEncoder(handle_unknown='ignore')), cat_cols)])
knn_flag = make_pipeline(prep_flag, KNeighborsClassifier(n_neighbors=5)).fit(X_train, y_train)
names_flag = prep_flag.get_feature_names_out()
print("n columns:", len(names_flag))
print("new column(s):", [n for n in names_flag if 'missingindicator' in n.lower()])
print(f"k-NN with missing flag   test accuracy {knn_flag.score(X_test, y_test):.3f}  (vs. 0.821 without it)")

n columns: 13
new column(s): ['num__missingindicator_Age']


k-NN with missing flag   test accuracy 0.821  (vs. 0.821 without it)


**Resposta:** Com `add_indicator=True`, a saída passa a ter 13 colunas em vez de 12 — uma a mais. A nova
coluna se chama `num__missingindicator_Age` (segundo `get_feature_names_out()`), porque `Age` é a única das quatro
colunas numéricas (`Age`, `SibSp`, `Parch`, `Fare`) que tinha valores faltantes no treino (137 buracos); o
indicador é 1 nas linhas em que a idade foi imputada e 0 nas demais. A acurácia do k-NN do passo 8 não muda nada
com esse sinalizador extra: 0.821 com a coluna indicadora, exatamente igual aos 0.821 sem ela — nenhum passageiro
do teste troca de classificação. Isso sugere que o próprio `KNeighborsClassifier`, que mede distância entre todas
as colunas, já não ganha nada ao saber explicitamente quais idades foram imputadas, quando as outras 12 colunas
(incluindo a própria idade imputada) já carregam informação suficiente.

In [18]:
df_titles = df.copy()
df_titles['Title'] = df_titles['Name'].str.extract(r',\s*([^.]+)\.')
features_title = features + ['Title']
cat_cols_title = cat_cols + ['Title']
Xt, yt = df_titles[features_title], df_titles['Survived']
Xt_train, Xt_test, yt_train, yt_test = train_test_split(Xt, yt, test_size=0.2, stratify=yt, random_state=42)

title_median = Xt_train.groupby('Title')['Age'].median()
global_median = X_train['Age'].median()
master_mask_train = (Xt_train['Title'] == 'Master') & Xt_train['Age'].isna()
print("n Master boys with missing age (train):", int(master_mask_train.sum()))
print("median age for title 'Master':", title_median.get('Master'))
print("global median (step 4):", round(global_median, 1))

ohe_title = OneHotEncoder(handle_unknown='ignore', min_frequency=10).fit(
    SimpleImputer(strategy='most_frequent').set_output(transform='pandas').fit_transform(Xt_train[cat_cols_title]))
print("one-hot columns with titles grouped (min_frequency=10):", len(ohe_title.get_feature_names_out()))

n Master boys with missing age (train): 3
median age for title 'Master': 3.0
global median (step 4): 28.5


one-hot columns with titles grouped (min_frequency=10): 13


**Resposta:** Há exatamente 3 meninos com título "Master" e idade faltante no treino (o mesmo número que
o enunciado da página antecipa). A mediana de idade do grupo "Master" é 3.0 anos — bem diferente da mediana global
de 28.5 usada no passo 4, uma diferença de 25.5 anos. Isso corrige uma distorção grave do passo 4: lá, esses três
meninos receberiam 28.5 anos (idade de adulto) só porque a imputação ignorava que "Master" é o título usado para
meninos, não para homens adultos; imputar pelo título devolve a eles uma idade de criança, coerente com a própria
informação que o nome completo já continha. Separadamente, o `OneHotEncoder(handle_unknown='ignore',
min_frequency=10)` agrupa os títulos raros (a maioria dos 17 títulos distintos no dataset tem poucos passageiros,
como `Dr` com 7) numa categoria "infrequent", produzindo 13 colunas one-hot no total para `Sex` + `Embarked` +
`Pclass` + `Title` juntos — bem menos do que se cada um dos 17 títulos virasse sua própria coluna.

In [19]:
preprocess_all = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                          OneHotEncoder(handle_unknown='ignore')), cat_cols)]).fit(X)

med_train = preprocess.named_transformers_['num'].named_steps['simpleimputer'].statistics_[num_cols.index('Age')]
med_all = preprocess_all.named_transformers_['num'].named_steps['simpleimputer'].statistics_[num_cols.index('Age')]

mean_fare_train = X_train['Fare'].mean()
std_fare_train = X_train['Fare'].std()
mean_fare_all = X['Fare'].mean()
std_fare_all = X['Fare'].std()

print(f"median Age   train-only {med_train:.2f}  ·  fit on all 891 rows {med_all:.2f}")
print(f"mean  Fare   train-only {mean_fare_train:.2f}  ·  fit on all 891 rows {mean_fare_all:.2f}")
print(f"std   Fare   train-only {std_fare_train:.2f}  ·  fit on all 891 rows {std_fare_all:.2f}")

median Age   train-only 28.50  ·  fit on all 891 rows 28.00
mean  Fare   train-only 31.82  ·  fit on all 891 rows 32.20
std   Fare   train-only 48.06  ·  fit on all 891 rows 49.69


**Resposta:** Ajustar o `preprocess` nas 891 linhas inteiras (em vez de só nas 712 de treino) é exatamente
o vazamento descrito no passo 2. Em termos absolutos, a mediana de `Age` se desloca de 28.50 para 28.00 (-0.50,
-1.8%); a média de `Fare` vai de 31.82 para 32.20 (+0.38, +1.2%); o desvio-padrão de `Fare` vai de 48.06 para
49.69 (+1.63, **+3.4%**) — em termos relativos, é o desvio-padrão de `Fare` que mais se move. Isso é coerente com
o fato de `Fare` ter uma cauda longa de valores extremos (até 512.33, como vimos no passo "To close"): o
desvio-padrão é a estatística mais sensível a outliers (ele cresce com o quadrado do afastamento de cada ponto),
então incluir os 179 passageiros de teste — com suas próprias tarifas altas — no ajuste desloca o desvio mais do
que desloca uma mediana, que é robusta por construção e quase não reage à entrada de mais linhas numa distribuição
já estável.

## Conclusões

- O vazamento de informação é concreto e mensurável: ajustar as estatísticas de pré-processamento no conjunto
  inteiro (891 linhas) em vez de só no treino (712 linhas) move a média/desvio de `Fare` muito mais do que a
  mediana de `Age` — a cauda longa de tarifas (até 512.33) é o que mais sofre com a mistura treino+teste.
- Em unidades brutas, `Fare` domina 93.1% da distância ao quadrado entre dois passageiros; depois do
  `StandardScaler`, as quatro colunas numéricas ficam igualadas em 25% cada — e essa decisão de escalar vale muito
  mais (+14.5 pontos percentuais de acurácia do k-NN, de 0.676 para 0.821) do que a escolha de qual escalador usar
  (StandardScaler 0.821, MinMaxScaler 0.804, RobustScaler 0.771).
- A "lei do encolhimento" (shrinkage law) da imputação por média se confirmou nas três rodadas de simulação (20%,
  50% e 80% de buracos artificiais), prevendo o desvio-padrão pós-imputação com duas casas decimais de precisão em
  todos os casos — e mostrando que, a 80% de buracos, o desvio cai para cerca de 6, menos da metade do valor real
  de 14.49.
- Corrigir a forma de uma distribuição (assimetria de `Fare` caindo de 4.64 para 0.31 com `log1p`) não é o mesmo
  que melhorar o modelo: a acurácia do k-NN caiu de 0.821 para 0.782 depois do log, porque a transformação também
  comprimiu o sinal útil que vinha das tarifas altas.
- Imputadores mais sofisticados (`KNNImputer`, `IterativeImputer`) produzem idades com desvio-padrão mais alto e
  mais realista (14.05 e 13.46, contra 13.03 da mediana, mais perto dos 14.49 observados) e evitam idades
  impossíveis só quando restringidos (`min_value=0`) — mas, mesmo assim, não melhoraram a acurácia final do k-NN
  (0.804 e 0.799 contra 0.821 da mediana simples), reforçando que "estatisticamente melhor" não é sinônimo de
  "modelo melhor".
- Em todo o notebook, `fit`/`fit_transform` só foi chamado sobre `X_train` (ou subconjuntos dele); o conjunto de
  teste só recebeu `transform` — a única exceção deliberada foi o exercício "the leak, in numbers", feito
  justamente para medir o tamanho do vazamento, não para validar um modelo.